# Chapter 12 — Intro to Recurrence: Predicting the Next Word

**Mata Kuliah:** Deep Learning
**Buku:** Grokking Deep Learning (Andrew W. Trask, Manning, 2019)
**Chapter 12:** Intro to Recurrence - Predicting the Next Word

## Isi Chapter 12

1. Tantangan panjang kalimat yang berubah-ubah
2. Rata-rata word vector dan kegunaannya
3. Keterbatasan bag of words: urutan kata hilang
4. Identity matrix dan penjumlahan embedding
5. Transition matrix yang dipelajari
6. Forward propagation dan backpropagation pada contoh kecil
7. Melatih RNN sederhana pada dataset bAbI
8. Perplexity dan membaca prediksi jaringan

## Catatan data

Notebook ini memakai tiga file dari repository resmi buku: `reviews.txt` dan `labels.txt`
(bagian 2) serta `tasksv11/en/qa1_single-supporting-fact_train.txt` (bagian 7). Sel
pertama mengunduhnya otomatis jika belum ada. Jalankan di Google Colab.

## 1. Tantangan panjang yang berubah-ubah

Di Chapter 11, embedding mewakili satu kata. Di chapter ini, targetnya adalah satu vektor
yang mewakili seluruh kalimat, apa pun panjangnya. Cara paling sederhana adalah menyambung
(concatenate) embedding semua kata. Masalahnya, `the cat sat` dan `the cat sat still`
menghasilkan vektor dengan panjang berbeda, sehingga keduanya tidak bisa dibandingkan
langsung. Kalau disejajarkan dari kiri, keduanya tampak identik; kalau dari kanan,
keduanya tampak sangat berbeda, padahal tiga dari empat katanya sama dan urutannya sama.

Alternatif yang lebih sederhana: rata-ratakan embedding semua kata. Panjang vektor kalimat
selalu sama dengan panjang embedding, jadi tidak ada masalah penyejajaran. Kalimat yang
kata-katanya mirip juga menghasilkan vektor yang mirip.

## 2. Kekuatan rata-rata word vector

Saya mencoba rata-rata embedding pada review IMDB. Agar notebook ini berdiri sendiri,
embedding saya latih dengan tugas klasifikasi sentimen dari Chapter 11 (embedding layer,
2 iterasi, beberapa detik), bukan dengan fill-in-the-blank yang butuh sekitar sepuluh
menit. Hasil pencarian review mirip nanti bisa sedikit berbeda dari embedding
fill-in-the-blank, tetapi cara kerjanya sama.

In [1]:
import os, urllib.request
import numpy as np
from collections import Counter

RAW = 'https://raw.githubusercontent.com/iamtrask/Grokking-Deep-Learning/{}/{}'

def ensure_file(name):
    if os.path.exists(name):
        return
    d = os.path.dirname(name)
    if d:
        os.makedirs(d, exist_ok=True)
    for branch in ['master', 'main']:          # cabang repo bisa master atau main
        try:
            urllib.request.urlretrieve(RAW.format(branch, name), name)
            return
        except Exception:
            pass
    raise RuntimeError('Gagal mengunduh ' + name)

ensure_file('reviews.txt')
ensure_file('labels.txt')
ensure_file('tasksv11/en/qa1_single-supporting-fact_train.txt')

f = open('reviews.txt'); raw_reviews = f.readlines(); f.close()
f = open('labels.txt'); raw_labels = f.readlines(); f.close()

tokens = list(map(lambda x: set(x.split(" ")), raw_reviews))

vocab = set()
for sent in tokens:
    for word in sent:
        if len(word) > 0:
            vocab.add(word)
vocab = sorted(vocab)   # diurutkan agar indeks kata sama setiap kali dijalankan

word2index = {}
for i, word in enumerate(vocab):
    word2index[word] = i

input_dataset = list()
for sent in tokens:
    sent_indices = list()
    for word in sent:
        try:
            sent_indices.append(word2index[word])
        except:
            ""
    input_dataset.append(list(set(sent_indices)))

target_dataset = list()
for label in raw_labels:
    if label == 'positive\n':
        target_dataset.append(1)
    else:
        target_dataset.append(0)

print('jumlah review     :', len(raw_reviews))
print('ukuran vocabulary :', len(vocab))

jumlah review     : 25000
ukuran vocabulary : 74074


Embedding dilatih dengan kode embedding layer dari Chapter 11 (24.000 review untuk
latihan, 1.000 terakhir untuk uji):

In [2]:
np.random.seed(1)

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

alpha, iterations = (0.01, 2)
hidden_size = 100
weights_0_1 = 0.2 * np.random.random((len(vocab), hidden_size)) - 0.1
weights_1_2 = 0.2 * np.random.random((hidden_size, 1)) - 0.1

for iter in range(iterations):
    correct, total = (0, 0)
    for i in range(len(input_dataset) - 1000):
        x, y = (input_dataset[i], target_dataset[i])

        layer_1 = sigmoid(np.sum(weights_0_1[x], axis=0))
        layer_2 = sigmoid(np.dot(layer_1, weights_1_2))

        layer_2_delta = layer_2 - y
        layer_1_delta = layer_2_delta.dot(weights_1_2.T)

        weights_0_1[x] -= layer_1_delta * alpha
        weights_1_2 -= np.outer(layer_1, layer_2_delta) * alpha

        if np.abs(layer_2_delta) < 0.5:
            correct += 1
        total += 1
    print('Iter:', iter, '| Training Accuracy:', round(correct / float(total), 4))

Iter: 0 | Training Accuracy: 0.8351
Iter: 1 | Training Accuracy: 0.9012


Berikutnya saya menormalkan embedding, membuat fungsi `make_sent_vect` yang merata-ratakan
embedding kata dalam sebuah review, lalu mencari tiga review yang paling mirip dengan
vektor rata-rata kata `boring` dan `awful` (dengan dot product).

Satu catatan tentang kode ini: baris normalisasi mengalikan `weights_0_1` dengan
`norms`, yaitu kuadrat panjang tiap vektor, bukan membaginya. Saya menulisnya apa adanya
dari kode buku. Kata dengan vektor panjang jadi punya bobot lebih besar saat dirata-rata.

In [3]:
norms = np.sum(weights_0_1 * weights_0_1, axis=1)
norms.resize(norms.shape[0], 1)
normed_weights = weights_0_1 * norms

def make_sent_vect(words):
    indices = list(map(lambda x: word2index[x],
                       filter(lambda x: x in word2index, words)))
    return np.mean(normed_weights[indices], axis=0)

reviews2vectors = list()
for review in tokens:                       # tokenized reviews
    reviews2vectors.append(make_sent_vect(review))
reviews2vectors = np.array(reviews2vectors)

def most_similar_reviews(review):
    v = make_sent_vect(review)
    scores = Counter()
    for i, val in enumerate(reviews2vectors.dot(v)):
        scores[i] = val
    most_similar = list()

    for idx, score in scores.most_common(3):
        most_similar.append(raw_reviews[idx][0:40])
    return most_similar

most_similar_reviews(['boring', 'awful'])

['the characters are unlikeable and the sc',
 'this movie is so bad  it can only be com',
 'this was one of the worst movies i have ']

Ketiga review yang muncul bernada negatif (`the characters are unlikeable...`,
`this movie is so bad...`, `this was one of the worst movies...`), padahal query hanya
berisi dua kata, `boring` dan `awful`. Embedding kata negatif mengelompok, sehingga
rata-ratanya mengarah ke review negatif. Jadi rata-rata embedding menyimpan informasi
statistik yang berguna.

## 3. Embedding sebagai garis berlekuk

Satu cara membayangkan vektor kata adalah garis berlekuk: nilai tinggi dan rendah di
posisi yang berbeda. Kata yang artinya mirip, misalnya `terrible` dan `boring`, punya
bentuk lekuk yang mirip. Lekuk itu tidak "berarti negatif" secara ajaib; ia berarti
negatif hanya karena semua kata negatif memilikinya, dan jika jaringan dilatih ulang
lekuk itu akan muncul di tempat lain.

Saat embedding banyak kata dirata-ratakan, lekuk yang dominan bertahan, sedangkan noise
dari kata-kata tertentu saling menghapus. Ini juga menjelaskan kelemahan pertama:
kalau terlalu banyak kata dirata-ratakan, hasilnya mendekati garis lurus (vektor hampir
nol), jadi hampir tidak ada informasi yang tersimpan. Penyimpanan informasi kalimat ini
tidak "meluruh dengan baik".

## 4. Keterbatasan bag of words

Kelemahan terbesar rata-rata embedding: urutan tidak punya arti. `Yankees defeat Red Sox`
dan `Red Sox defeat Yankees` menghasilkan vektor yang sama persis, padahal artinya
berlawanan. `Sox Red Yankees defeat` pun sama. Pendekatan ini disebut bag of words
karena kata-kata dimasukkan ke dalam "kantong" tanpa urutan. Penyebabnya sederhana:
penjumlahan itu komutatif (`a + b == b + a`).

Saya mencoba membuktikannya dengan vektor kecil:

In [4]:
# Percobaan tambahan (bukan dari kode buku): urutan tidak mengubah hasil penjumlahan
red    = np.array([1., 0., 2.])
sox    = np.array([0., 3., 1.])
defeat = np.array([2., 1., 0.])
yankees = np.array([1., 1., 1.])

kalimat_a = [yankees, defeat, red, sox]     # "yankees defeat red sox"
kalimat_b = [red, sox, defeat, yankees]     # "red sox defeat yankees"

print(sum(kalimat_a))
print(sum(kalimat_b))
print('sama persis:', np.array_equal(sum(kalimat_a), sum(kalimat_b)))

[4. 5. 4.]
[4. 5. 4.]
sama persis: True


Yang diinginkan adalah vektor kalimat yang berubah jika urutan berubah, dan cara urutan
mengubahnya sebaiknya dipelajari dari tugas bahasa. Salah satu cara paling terkenal
untuk itu adalah recurrent neural network (RNN). Untuk sampai ke sana, ada langkah
perantara yang tampak boros: menjumlahkan embedding sambil mengalikan dengan
*identity matrix*.

## 5. Identity matrix

Identity matrix adalah matriks persegi yang berisi 0, dengan 1 di sepanjang diagonal
(kiri atas ke kanan bawah). Perkalian vektor dengan identity matrix mengembalikan
vektor yang sama.

In [5]:
import numpy as np

a = np.array([1, 2, 3])
b = np.array([0.1, 0.2, 0.3])
c = np.array([-1, -0.5, 0])
d = np.array([0, 0, 0])

identity = np.eye(3)
print(identity)

[[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]


In [6]:
print(a.dot(identity))
print(b.dot(identity))
print(c.dot(identity))
print(d.dot(identity))

[1. 2. 3.]
[0.1 0.2 0.3]
[-1.  -0.5  0. ]
[0. 0. 0.]


Karena itu, menyisipkan perkalian identity matrix di antara penjumlahan tidak mengubah
hasil. Saya cek dengan tiga kata `this movie rocks`:

In [7]:
this = np.array([2, 4, 6])
movie = np.array([10, 10, 10])
rocks = np.array([1, 1, 1])

print(this + movie + rocks)
print((this.dot(identity) + movie).dot(identity) + rocks)

[13 15 17]
[13. 15. 17.]


Kedua cara menghasilkan vektor yang sama. Identity matrix adalah satu-satunya matriks
yang dijamin mengembalikan vektor masukannya. Jika matriks yang dipakai bukan identity,
vektor kalimat akan bergantung pada urutan kata. Saya coba dengan matriks acak:

In [8]:
# Percobaan tambahan (bukan dari kode buku): matriks bukan identity membuat urutan berpengaruh
np.random.seed(0)
m = np.random.rand(3, 3)

urutan_1 = (this.dot(m) + movie).dot(m) + rocks      # this -> movie -> rocks
urutan_2 = (rocks.dot(m) + movie).dot(m) + this      # rocks -> movie -> this

print(urutan_1)
print(urutan_2)
print('sama:', np.allclose(urutan_1, urutan_2))

[28.35872472 37.65368215 41.37878014]
[20.22776046 28.2344978  32.48969866]
sama: False


Urutan kata yang berbeda sekarang menghasilkan vektor yang berbeda
(`[28.4, 37.7, 41.4]` dan `[20.2, 28.2, 32.5]`). Matriks acak ini hanya untuk
demonstrasi; di jaringan sebenarnya matriks itu dipelajari, bukan diacak.

## 6. Transition matrix yang dipelajari

Pertanyaannya, matriks apa yang dipakai menggantikan identity? Jawaban standar di deep
learning: dipelajari seperti bobot lain. Matriks itu dimulai sebagai identity, lalu
ikut diperbarui oleh gradien. Tugas belajarnya adalah memprediksi kata berikutnya dari
kata-kata sebelumnya, mirip fill-in-the-blank di Chapter 11.

Ada satu batasan: semua transition matrix dibuat sama (matriks yang dipakai untuk
`Red -> Sox` dipakai ulang untuk `Sox -> defeat`). Dengan begitu, hanya logika yang
berguna di setiap langkah yang bisa dipelajari.

Jaringannya terdiri dari dua langkah: membuat sentence embedding dari kata-kata yang
sudah dilihat, lalu memakainya untuk memprediksi kata berikutnya lewat softmax atas
seluruh vocabulary.

## 7. Forward propagation pada contoh kecil

Saya mulai dari versi kecil dengan sembilan kata. Semua embedding diinisialisasi nol,
sedangkan `sent2output` (bobot dari sentence embedding ke output) acak.

In [9]:
import numpy as np

def softmax(x_):
    x = np.atleast_2d(x_)
    temp = np.exp(x)
    return temp / np.sum(temp, axis=1, keepdims=True)

word_vects = {}
word_vects['yankees'] = np.array([[0., 0., 0.]])
word_vects['bears'] = np.array([[0., 0., 0.]])
word_vects['braves'] = np.array([[0., 0., 0.]])
word_vects['red'] = np.array([[0., 0., 0.]])
word_vects['sox'] = np.array([[0., 0., 0.]])
word_vects['lose'] = np.array([[0., 0., 0.]])
word_vects['defeat'] = np.array([[0., 0., 0.]])
word_vects['beat'] = np.array([[0., 0., 0.]])
word_vects['tie'] = np.array([[0., 0., 0.]])

sent2output = np.random.rand(3, len(word_vects))

identity = np.eye(3)

In [10]:
layer_0 = word_vects['red']
layer_1 = layer_0.dot(identity) + word_vects['sox']
layer_2 = layer_1.dot(identity) + word_vects['defeat']

pred = softmax(layer_2.dot(sent2output))
print(pred)

[[0.11111111 0.11111111 0.11111111 0.11111111 0.11111111 0.11111111
  0.11111111 0.11111111 0.11111111]]


Semua probabilitas sama, yaitu 1/9 ≈ 0.1111. Semua embedding masih nol, sehingga
`layer_2` nol dan softmax atas sembilan kata memberi probabilitas seragam. Jaringan belum
belajar apa-apa.

## 8. Backpropagation ke dalam jaringan ini

Bagian ini tampak lebih rumit, padahal langkahnya sama dengan yang sudah dipelajari.
Dari `pred` sampai `layer_2_delta` adalah backpropagation biasa (seperti Chapter 9).
Setelah itu, gradien di `layer_2` harus dibagi dua arah, karena `layer_2` adalah hasil
penjumlahan: satu ke `word_vects['defeat']`, satu lagi melewati identity matrix ke
`layer_1`. Prinsipnya: ketika dua vektor dijumlahkan di forward propagation, gradien
yang sama dikirim ke kedua sisi penjumlahan saat backpropagation.

In [11]:
y = np.array([1, 0, 0, 0, 0, 0, 0, 0, 0])   # target: one-hot untuk "yankees"

pred_delta = pred - y
layer_2_delta = pred_delta.dot(sent2output.T)
defeat_delta = layer_2_delta * 1             # angka "1" boleh diabaikan seperti Chapter 11
layer_1_delta = layer_2_delta.dot(identity.T)
sox_delta = layer_1_delta * 1                # sama, boleh diabaikan
layer_0_delta = layer_1_delta.dot(identity.T)
alpha = 0.01
word_vects['red'] -= layer_0_delta * alpha
word_vects['sox'] -= sox_delta * alpha
word_vects['defeat'] -= defeat_delta * alpha
identity -= np.outer(layer_0, layer_1_delta) * alpha
identity -= np.outer(layer_1, layer_2_delta) * alpha
sent2output -= np.outer(layer_2, pred_delta) * alpha

print(word_vects['defeat'])   # tambahan: embedding yang tadinya nol sekarang berubah

[[-0.00084193  0.00159323  0.00435671]]


Satu langkah update sudah mengubah embedding `red`, `sox`, dan `defeat` dari nol menjadi
tidak nol (output di atas menunjukkan `defeat`). Embedding `yankees` tetap nol karena ia
target, bukan bagian input.

`identity` juga ikut berubah, tetapi lewat jalur yang halus: `layer_0` menunjuk ke array
`word_vects['red']` yang sudah diubah di tempat (`-=`) pada baris sebelumnya, jadi nilainya
sudah tidak nol saat dipakai untuk meng-update `identity`. Sebaliknya `layer_1` dan
`layer_2` masih nol, sehingga bagian update `identity` dari `layer_1` dan update
`sent2output` belum menambah perubahan apa pun pada langkah pertama ini. Pelatihan
sebenarnya mengulang langkah seperti ini ribuan kali.

## 9. Melatih pada dataset bAbI

Sekarang jaringan dilatih pada dataset kecil, bAbI: kumpulan pertanyaan-jawaban
sintetis tentang lingkungan sederhana (siapa pindah ke mana). Di sini dataset tidak
dipakai untuk menjawab pertanyaan, hanya untuk menyelesaikan kalimat dari satu kata
awal atau lebih. Karena polanya sederhana, efek dari transition matrix yang dipelajari
lebih mudah terlihat.

Saya ambil 1.000 baris pertama, buang nomor barisnya, dan huruf kecilkan.

In [12]:
import sys, random, math
from collections import Counter
import numpy as np

f = open('tasksv11/en/qa1_single-supporting-fact_train.txt', 'r')
raw = f.readlines()
f.close()

tokens = list()
for line in raw[0:1000]:
    tokens.append(line.lower().replace("\n", "").split(" ")[1:])

print(tokens[0:3])

[['mary', 'moved', 'to', 'the', 'bathroom.'], ['john', 'went', 'to', 'the', 'hallway.'], ['where', 'is', 'mary?', '\tbathroom\t1']]


Dataset berisi pernyataan sederhana dan pertanyaan (tanda tanya tetap menempel pada kata,
dan pertanyaan diikuti jawaban dan nomor baris yang ikut menjadi token).

Berikutnya saya membuat vocabulary dan fungsi pembantu. Vocabulary diurutkan
(`sorted`) agar hasilnya sama setiap dijalankan; di kode buku dibuat dengan
`list(set(...))`, yang urutannya bisa berubah antar sesi.

In [13]:
vocab = set()
for sent in tokens:
    for word in sent:
        vocab.add(word)

vocab = sorted(vocab)

word2index = {}
for i, word in enumerate(vocab):
    word2index[word] = i

def words2indices(sentence):
    idx = list()
    for word in sentence:
        idx.append(word2index[word])
    return idx

def softmax(x):
    e_x = np.exp(x - np.max(x))
    return e_x / e_x.sum(axis=0)

print('ukuran vocabulary:', len(vocab))

ukuran vocabulary: 82


Inisialisasi bobot. `embed_size = 10`. Ada:

- `embed`: embedding kata,
- `recurrent`: transition matrix, dimulai sebagai identity matrix,
- `start`: sentence embedding untuk kalimat kosong, yang penting agar jaringan
  belajar bagaimana kalimat biasanya dimulai,
- `decoder`: bobot dari embedding ke output (sama seperti di embedding sebelumnya),
- `one_hot`: tabel untuk menghitung loss.

In [14]:
np.random.seed(1)
embed_size = 10

# word embeddings
embed = (np.random.rand(len(vocab), embed_size) - 0.5) * 0.1

# embedding -> embedding (initially the identity matrix)
recurrent = np.eye(embed_size)

# sentence embedding for empty sentence
start = np.zeros(embed_size)

# embedding -> output weights
decoder = (np.random.rand(embed_size, len(vocab)) - 0.5) * 0.1

# one hot lookups (for loss function)
one_hot = np.eye(len(vocab))

## 10. Forward propagation dengan panjang berubah-ubah

Fungsi `predict` melakukan forward propagation: di setiap langkah waktu, jaringan
memprediksi kata berikutnya dari `hidden` saat ini (`layer['pred']`), menambah loss,
lalu membuat `hidden` berikutnya dengan `hidden.dot(recurrent) + embed[kata]`.
Prediksi dibuat di setiap langkah, bukan hanya di akhir, sehingga tidak perlu
mengulang forward propagation dari awal untuk setiap kata.

Karena panjang kalimat berbeda-beda, layer tidak bisa berupa variabel tetap. Layer
disimpan dalam list `layers`, dan list ini bertambah sesuai panjang kalimat. Saya
perlu memahami isi list ini dengan baik karena backpropagation dan update bobot
memakainya.

In [15]:
def predict(sent):

    layers = list()
    layer = {}
    layer['hidden'] = start
    layers.append(layer)

    loss = 0

    # forward propagate
    preds = list()
    for target_i in range(len(sent)):

        layer = {}

        # try to predict the next term
        layer['pred'] = softmax(layers[-1]['hidden'].dot(decoder))

        loss += -np.log(layer['pred'][sent[target_i]])

        # generate the next hidden state
        layer['hidden'] = layers[-1]['hidden'].dot(recurrent) + embed[sent[target_i]]
        layers.append(layer)

    return layers, loss

## 11. Backpropagation dan update bobot dengan panjang berubah-ubah

Backpropagation dilakukan dari layer terakhir ke awal. Untuk setiap layer (kecuali yang
pertama), `output_delta = pred - one_hot[target]` lalu `new_hidden_delta` dihitung lewat
`decoder`. Pada layer terakhir, `hidden_delta` hanya berisi `new_hidden_delta`, karena
tidak ada layer sesudahnya. Pada layer lain, ditambah gradien dari layer berikutnya yang
dikirim mundur lewat `recurrent`. Layer pertama (kalimat kosong) hanya menerima gradien
dari layer berikutnya.

Update bobot memakai gradien itu, dibagi `len(sent)` supaya kalimat yang panjang tidak
membuat langkah update jadi terlalu besar. Setiap seribu iterasi, saya mencetak
perplexity.

Tambahan saya (bukan di kode buku): pada iterasi 100 dan 10.000 saya mencetak prediksi
untuk kalimat ke-4, agar tahap-tahap belajar jaringan terlihat. Bagian itu ada di akhir
loop, diberi komentar.

In [16]:
def show_predictions(sent_index):
    l, _ = predict(words2indices(tokens[sent_index]))
    print(tokens[sent_index])
    for i, each_layer in enumerate(l[1:-1]):
        input = tokens[sent_index][i]
        true = tokens[sent_index][i+1]
        pred = vocab[each_layer['pred'].argmax()]
        print("Prev Input:" + input + (' ' * (12 - len(input))) +
              "True:" + true + (" " * (15 - len(true))) + "Pred:" + pred)

# forward
for iter in range(30000):
    alpha = 0.001
    sent = words2indices(tokens[iter % len(tokens)][1:])

    layers, loss = predict(sent)

    # back propagate
    for layer_idx in reversed(range(len(layers))):
        layer = layers[layer_idx]
        target = sent[layer_idx-1]

        if(layer_idx > 0):
            layer['output_delta'] = layer['pred'] - one_hot[target]
            new_hidden_delta = layer['output_delta'].dot(decoder.transpose())

            # if the last layer - don't pull from a
            # later one because it doesn't exist
            if(layer_idx == len(layers)-1):
                layer['hidden_delta'] = new_hidden_delta
            else:
                layer['hidden_delta'] = new_hidden_delta + layers[layer_idx+1]['hidden_delta'].dot(recurrent.transpose())
        else:
            layer['hidden_delta'] = layers[layer_idx+1]['hidden_delta'].dot(recurrent.transpose())

    # update weights
    start -= layers[0]['hidden_delta'] * alpha / float(len(sent))
    for layer_idx, layer in enumerate(layers[1:]):

        decoder -= np.outer(layers[layer_idx]['hidden'], layer['output_delta']) * alpha / float(len(sent))

        embed_idx = sent[layer_idx]
        embed[embed_idx] -= layers[layer_idx]['hidden_delta'] * alpha / float(len(sent))
        recurrent -= np.outer(layers[layer_idx]['hidden'], layer['hidden_delta']) * alpha / float(len(sent))

    if(iter % 1000 == 0):
        print("Perplexity:" + str(np.exp(loss/len(sent))))

    # tambahan: lihat prediksi pada iterasi 100 dan 10.000
    if iter in (100, 10000):
        print('--- prediksi setelah', iter, 'iterasi ---')
        show_predictions(4)

Perplexity:81.94009027613271
--- prediksi setelah 100 iterasi ---
['sandra', 'moved', 'to', 'the', 'garden.']
Prev Input:sandra      True:moved          Pred:	bedroom	14
Prev Input:moved       True:to             Pred:	hallway	8
Prev Input:to          True:the            Pred:	hallway	8
Prev Input:the         True:garden.        Pred:	hallway	8
Perplexity:81.85534314118429
Perplexity:81.71905473544616
Perplexity:81.45316435924049
Perplexity:80.9044869625085
Perplexity:79.71575940161105
Perplexity:76.84808386902766
Perplexity:67.70605588988157
Perplexity:37.525750371346206
Perplexity:22.332938427124084
Perplexity:19.54837507041369
--- prediksi setelah 10000 iterasi ---
['sandra', 'moved', 'to', 'the', 'garden.']
Prev Input:sandra      True:moved          Pred:the
Prev Input:moved       True:to             Pred:the
Prev Input:to          True:the            Pred:the
Prev Input:the         True:garden.        Pred:the
Perplexity:18.372526919748296
Perplexity:17.04665571370902
Perplexity:1

Perplexity awal 81.94, hampir sama dengan ukuran vocabulary (82), artinya jaringan
menebak hampir seragam. Perplexity nyaris tidak bergerak sampai iterasi 5.000
(81.9 → 79.7), lalu turun tajam: 76.8 (6.000), 67.7 (7.000), 37.5 (8.000), 22.3
(9.000), dan akhirnya 4.20 pada iterasi 29.000.

Prediksi memperlihatkan urutan belajarnya. Pada iterasi 100, tebakan masih acak (potongan
seperti `\tbedroom\t14`, yaitu bagian jawaban dari baris pertanyaan). Pada iterasi 10.000,
jaringan menebak `the` di semua posisi: tebakan yang aman untuk kata yang sering muncul.
Detail yang lebih halus baru dipelajari setelah itu.

## 12. Membaca prediksi jaringan

Perplexity adalah probabilitas kata yang benar, dilewatkan log, dinegatifkan, dan
dieksponensialkan (e^x). Secara teori, ini mengukur perbedaan antara dua distribusi
probabilitas: distribusi sempurna (100% di kata yang benar) dan prediksi jaringan.
Perplexity tinggi bila distribusinya berbeda jauh, dan mendekati 1 bila cocok. Seperti
loss lain pada SGD, turunnya perplexity berarti prediksi jaringan makin sesuai data.

Angka itu sendiri belum menceritakan apa yang terjadi di dalam bobot, jadi saya lihat
prediksinya langsung pada kalimat ke-4 setelah training selesai:

In [17]:
sent_index = 4

l, _ = predict(words2indices(tokens[sent_index]))

print(tokens[sent_index])

for i, each_layer in enumerate(l[1:-1]):
    input = tokens[sent_index][i]
    true = tokens[sent_index][i+1]
    pred = vocab[each_layer['pred'].argmax()]
    print("Prev Input:" + input + (' ' * (12 - len(input))) +
          "True:" + true + (" " * (15 - len(true))) + "Pred:" + pred)

['sandra', 'moved', 'to', 'the', 'garden.']
Prev Input:sandra      True:moved          Pred:is
Prev Input:moved       True:to             Pred:to
Prev Input:to          True:the            Pred:the
Prev Input:the         True:garden.        Pred:bedroom.


Dua catatan untuk membaca output ini, yang saya cek langsung dengan kode:

1. Kolom `Pred` diambil dari `l[1:-1]`. `layers[k]['pred']` adalah tebakan untuk kata
   ke-(k-1) dari kata-kata sebelumnya, jadi `Pred` pada baris `sandra` adalah tebakan
   untuk kata pertama dari kalimat kosong (hasilnya sama dengan tebakan dari state
   awal, yaitu `is`). Pada baris lain, `Pred` adalah tebakan untuk kata di baris itu,
   bukan kata sesudahnya.
2. Saat training, `[1:]` membuang kata pertama (nama orang), jadi jaringan hanya pernah
   melihat urutan yang dimulai dari kata kerja, seperti `moved to the garden.`

Karena itu, output ini lebih tepat dibaca sebagai gambaran kebiasaan jaringan, bukan
skor ketepatan. Untuk melihat pasangan "kata-kata sebelumnya → tebakan kata berikutnya"
secara sejajar:

In [18]:
# Tambahan (bukan dari kode buku): tebakan kata berikutnya dari setiap awalan kalimat
l, _ = predict(words2indices(tokens[sent_index]))
print('dari kalimat kosong ->', vocab[l[1]['pred'].argmax()])
for i, each_layer in enumerate(l[2:]):
    print(tokens[sent_index][:i+1], '->', vocab[each_layer['pred'].argmax()],
          '| kata sebenarnya:', tokens[sent_index][i+1])

dari kalimat kosong -> is
['sandra'] -> to | kata sebenarnya: moved
['sandra', 'moved'] -> the | kata sebenarnya: to
['sandra', 'moved', 'to'] -> bedroom. | kata sebenarnya: the
['sandra', 'moved', 'to', 'the'] -> bedroom. | kata sebenarnya: garden.


Pada tampilan sejajar, setelah training selesai keempat tebakan bergeser satu posisi dari
kata sebenarnya: setelah `sandra` jaringan menebak `to`, setelah `sandra moved` menebak
`the`, lalu `bedroom.` dua kali. Urutan `to`, `the`, nama tempat adalah pola yang sama
dengan kalimat saat training (`moved to the garden.`), jadi hasil ini konsisten dengan
jaringan yang memperlakukan `sandra` seperti kata kerja pertama, karena saat training
kata pertama dibuang. Bukan kegagalan RNN, melainkan akibat cara data disiapkan.
`bedroom.` sendiri sama-sama nama tempat seperti `garden.`, jadi kesalahannya masuk akal.

Tugas ini juga hampir mustahil diselesaikan sempurna. Dari `sandra moved to the` saja,
tidak ada cukup konteks untuk tahu tempat yang benar. Justru karena tidak terselesaikan,
kesalahannya membantu melihat apa yang sebenarnya dipelajari jaringan.

## 13. Urutan sekarang berpengaruh

Sebagai penutup, saya cek apakah tujuan chapter tercapai: vektor kalimat yang peka
terhadap urutan. Saya bandingkan dua cara membuat vektor untuk urutan kata yang sama
tetapi dibalik: menjumlahkan embedding (bag of words) dan mengalirkan embedding lewat
`recurrent` yang sudah dilatih.

In [19]:
# Percobaan tambahan (bukan dari kode buku)
def sentence_vector(words):
    h = start
    for w in words:
        h = h.dot(recurrent) + embed[word2index[w]]
    return h

def bag_vector(words):
    return sum(embed[word2index[w]] for w in words)

def cosine(u, v):
    return u.dot(v) / (np.linalg.norm(u) * np.linalg.norm(v))

maju = ['mary', 'moved', 'to', 'the']
balik = list(reversed(maju))

print('bag of words   sama persis :', np.allclose(bag_vector(maju), bag_vector(balik)))
print('RNN            sama persis :', np.allclose(sentence_vector(maju), sentence_vector(balik)))
print('RNN cosine(maju, balik)    :', round(float(cosine(sentence_vector(maju), sentence_vector(balik))), 4))
print('jarak recurrent dari identity:', round(float(np.linalg.norm(recurrent - np.eye(embed_size))), 4))

bag of words   sama persis : True
RNN            sama persis : False
RNN cosine(maju, balik)    : 0.9982
jarak recurrent dari identity: 5.3624


Bag of words menghasilkan vektor yang sama persis untuk kedua urutan, sedangkan
vektor dari RNN tidak sama. Jadi urutan sekarang mengubah vektor kalimat. Namun cosine
keduanya masih 0.9982, artinya arah kedua vektor tetap sangat mirip, meskipun
`recurrent` sudah bergeser jauh dari identity (jarak 5.36). Pada percobaan sekecil ini,
pengaruh urutan ada tetapi kecil dalam ukuran cosine; saya tidak menyelidiki penyebabnya
lebih jauh.

## Key Ideas — Chapter 12

- Vektor kalimat dengan panjang tetap dibutuhkan agar kalimat berbeda panjang bisa dibandingkan.
- Rata-rata word embedding adalah cara sederhana yang cukup kuat, tetapi urutan kata hilang (bag of words).
- Kalau terlalu banyak kata dirata-ratakan, vektor mendekati nol dan informasinya habis.
- Perkalian dengan identity matrix tidak mengubah apa pun; menggantinya dengan matriks lain membuat urutan berpengaruh.
- Transition matrix dimulai sebagai identity, dipakai bersama di semua langkah, dan dipelajari lewat tugas memprediksi kata berikutnya.
- Saat dua vektor dijumlahkan, gradien yang sama dikirim ke kedua sisinya saat backpropagation.
- Prediksi dibuat di setiap langkah waktu, dan layer disimpan dalam list karena panjang kalimat berubah.
- Perplexity yang turun menandakan prediksi makin sesuai data, tetapi membaca prediksi langsung memberi gambaran lebih jelas.

## Kesimpulan

Chapter 12 memperkenalkan recurrent neural network melalui langkah-langkah kecil:
dari rata-rata embedding, ke penjumlahan dengan identity matrix, hingga transition
matrix yang dipelajari. Tidak ada konsep baru dalam backpropagation-nya; yang baru
adalah menyimpan state di setiap langkah waktu dan mengalirkan gradien mundur lewat
`recurrent`. Hasilnya, vektor kalimat yang peka terhadap urutan kata. Chapter berikutnya
membahas automatic differentiation untuk membangun framework deep learning sendiri.